# Session 4 — Frequency Analysis, Convolution, and Deconvolution

We will cover the following topics in this tutorial:
- How an image can be represented in the frequency domain
- How to perform convolution in the spatial domain and in the frequency domain
- How to perform deconvolution

## Now we will create some test images and see their frequency domain representation

**Ex 1:** Change the `frequency` and `orientation` parameters in the next cell and re-run it. Check that the resulting pattern matches what you'd expect, and explain what changing each parameter does to the image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Define the size of the image
width = 64
height = 64
frequency = 1 / 10  # the frequency is 1/L where L is the number of pixels between two peaks
orientation = 45  # in degrees

# Create a grid of coordinates (arange gives exact, evenly-spaced pixel coordinates)
x = np.arange(width)
y = np.arange(height)
X, Y = np.meshgrid(x, y)
U = X * np.cos(orientation * 2 * np.pi / 360) + Y * np.sin(orientation * 2 * np.pi / 360)  # rotate by orientation

# Create the sinusoidal pattern
# For the pattern to tile exactly (matching the periodic boundary the DFT assumes), choose
# 1/frequency to evenly divide width/height, otherwise there is a visible seam at the edges.
image = 255 * np.sin(U * 2 * np.pi * frequency)

# Display the image
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()

## Now, let's compute the Fourier representation from scratch and compare with the numpy fft2 function

In [ ]:
import time

import numpy as np


# Ex 2: Implement the 2D Discrete Fourier Transform (DFT) from scratch
# implement fourier_transform_2d(image) -> (complex) image
# TODO: your code here


# This is an O(N^4) implementation: fine for teaching, far too slow to call interactively.
# We run it ONCE, on a small 32x32 crop, and reuse the result below (see Ex 4).
image_crop = image[:32, :32]
t0 = time.perf_counter()
F_naive = fourier_transform_2d(image_crop)
t_naive = time.perf_counter() - t0
print(f"Naive O(N^4) DFT on a 32x32 crop took {t_naive:.2f}s")

In [ ]:
# Ex 3: Change the frequency and orientation parameters in the cell above and re-run this
# cell; explain what you see in the frequency-domain plot below.
def show_fourier_transform(image):
    height, width = image.shape
    plt.imshow(np.log1p(np.abs(np.fft.fftshift(image))), cmap="jet")
    plt.arrow(width / 2, height / 2, 0, 10, color="red", head_width=2)  # mark the zero frequency
    plt.arrow(width / 2, height / 2, 10, 0, color="red", head_width=2)  # mark the zero frequency
    plt.axis("off")
    plt.show()


# We use the fast FFT here for interactive exploration; the from-scratch version above is
# validated against it in Ex 4.
F_image = np.fft.fft2(image)
show_fourier_transform(F_image)

## Our implementation is easy to read, but takes quite some time — compare its time with np.fft.fft2

**Ex 4:** Compare running time and results with the optimized `fft2` function from NumPy.

In [ ]:
# Reuse F_naive and t_naive computed once above (same 32x32 crop) — do not re-run the O(N^4) loop.
# TODO: your code here

## Look at the Fourier representation of a natural image

In [ ]:
# We will get a test image from scikit-image
from skimage import data

image = data.cat()
plt.imshow(image)
plt.axis("off")
plt.title("Test image")
plt.show()

### Filter the high frequencies of the image

In [ ]:
# Since it's more efficient, from now on we will use fft2 from numpy to compute the Fourier transform
from numpy.fft import fft2, ifft2, fftshift, ifftshift


def ideal_lowpass_mask(shape, threshold):
    # A square "brick-wall" mask: 1 inside a (2*threshold)-wide box centered at the zero
    # frequency, 0 outside. Square masks are simple but cause ringing (Gibbs artifacts) at
    # sharp edges; a circular mask (or a smooth roll-off) reduces ringing but is harder to
    # reason about by hand.
    H, W = shape
    mask = np.zeros(shape)
    mask[int(H / 2) - threshold : int(H / 2) + threshold, int(W / 2) - threshold : int(W / 2) + threshold] = 1
    return mask


# Ex 5: Implement a low-pass filter in the Fourier domain
# filtered_image = low_pass_filter(image, threshold)
# Tip: define the FT (and its inverse) with the following shortcuts:
# def FT(I): return fftshift(fft2(I))
# def iFT(I): return ifft2(ifftshift(I))
# TODO: your code here

In [ ]:
filtered_image = low_pass_filter(image, 50)
# Display the original image and the filtered image
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.axis("off")
plt.title("Original image")
plt.subplot(1, 2, 2)
plt.imshow(filtered_image)
plt.axis("off")
plt.title("Filtered image")
plt.show()

### Filter the low frequencies

In [ ]:
# Ex 6: Implement a high-pass filter in the Fourier domain
# filtered_image = high_pass_filter(image, threshold)
# Tip: the high-pass mask is the complement of the low-pass mask above
# TODO: your code here

In [ ]:
filtered_image = high_pass_filter(image, 4)
# Display the original image and the filtered image
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.axis("off")
plt.title("Original image")
plt.subplot(1, 2, 2)
plt.imshow(filtered_image)
plt.axis("off")
plt.title("Filtered image")
plt.show()

## Apply similar filters using convolutions in the spatial domain, and compare

Filtering an image is a convolution: the frequency-domain masking we did above is exactly equivalent to convolving the image with the mask's inverse Fourier transform (a spatial kernel). Below we build a few classic spatial-domain filters directly with small kernels.

**Ex 7:** Apply a low-pass filter using convolution with a Gaussian kernel, and compare it with the frequency-domain version above.

In [ ]:
import cv2


# low_pass_conv(image, sigma) -> filtered image
# Tip: cv2.GaussianBlur(image, ksize, sigma) convolves the image with a Gaussian kernel; a
# wider kernel (larger sigma) removes more high-frequency detail.
# TODO: your code here


filtered_conv = low_pass_conv(image, sigma=5)
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.axis("off")
plt.title("Original")
plt.subplot(1, 2, 2)
plt.imshow(filtered_conv)
plt.axis("off")
plt.title("Low-pass (Gaussian conv.)")
plt.show()

**Ex 8:** Apply a band-pass filter using convolution with a kernel that is the difference of two Gaussians (DoG).

In [ ]:
# band_pass_conv(image, sigma1, sigma2) -> filtered image, with sigma1 < sigma2
# Tip: DoG = GaussianBlur(image, sigma1) - GaussianBlur(image, sigma2); it keeps the frequency
# band between the two cutoffs and is a classic edge/blob-enhancement operator.
# TODO: your code here


band = band_pass_conv(image, sigma1=1, sigma2=5)
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.axis("off")
plt.title("Original")
plt.subplot(1, 2, 2)
plt.imshow(band)
plt.axis("off")
plt.title("Band-pass (Difference of Gaussians)")
plt.show()

**Ex 9:** Apply filters that compute the x and y partial derivatives, via convolution.

In [ ]:
# partial_derivatives(image) -> (dx, dy)
# Tip: convolve a grayscale version of the image with a simple [-1, 0, 1] kernel (or cv2.Sobel)
# TODO: your code here


dx, dy = partial_derivatives(image)
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(image)
plt.axis("off")
plt.title("Original")
plt.subplot(1, 3, 2)
plt.imshow(dx, cmap="gray")
plt.axis("off")
plt.title("d/dx")
plt.subplot(1, 3, 3)
plt.imshow(dy, cmap="gray")
plt.axis("off")
plt.title("d/dy")
plt.show()

## Deconvolution

A blurred image can often be modeled as the convolution of a sharp image `f` with a blur kernel (point-spread function, PSF) `h`, plus noise: `g = f * h + n`. **Deconvolution** tries to invert this — given `g` and `h`, recover `f`.

Convolution becomes multiplication in the frequency domain (`G = F · H`), so the obvious fix is to divide: `F̂ = G / H`. That works perfectly when there is no noise. But real images always have some noise, and that noise gets divided by `H` too. A blur kernel is a low-pass filter, so `H` is small (near zero) at high frequencies — dividing by a near-zero number amplifies whatever noise lives there, often catastrophically. **Wiener filtering** regularizes the division to trade off deblurring against noise amplification.

In [ ]:
# Build a blurred + noisy version of the cat image to deconvolve.
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(float)
H, W = gray.shape

# Blur kernel (point-spread function): a small 2D Gaussian
psf_sigma = 3
ksize = 2 * int(np.ceil(3 * psf_sigma)) + 1
kernel_1d = cv2.getGaussianKernel(ksize, psf_sigma)
psf = kernel_1d @ kernel_1d.T

# Place the kernel at the center of a full-size canvas, then shift its center to (0, 0) (with
# wraparound) so that multiplying in the frequency domain matches a *circular* convolution.
psf_padded = np.zeros((H, W))
top, left = (H - ksize) // 2, (W - ksize) // 2
psf_padded[top : top + ksize, left : left + ksize] = psf
psf_padded = np.fft.ifftshift(psf_padded)
F_psf = np.fft.fft2(psf_padded)

# Blur the image (multiplication in frequency domain = convolution in space) and add noise
F_gray = np.fft.fft2(gray)
blurred = np.real(np.fft.ifft2(F_gray * F_psf))
noisy_blurred = blurred + np.random.normal(0, 2, blurred.shape)
F_blurred = np.fft.fft2(noisy_blurred)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.axis("off")
plt.title("Original")
plt.subplot(1, 3, 2)
plt.imshow(blurred, cmap="gray")
plt.axis("off")
plt.title("Blurred")
plt.subplot(1, 3, 3)
plt.imshow(noisy_blurred, cmap="gray")
plt.axis("off")
plt.title("Blurred + noisy")
plt.show()

In [ ]:
# Naive inverse filter: divide out the blur in the frequency domain, F_hat = G / H
epsilon = 1e-3  # avoid literal division by zero
F_naive_inv = F_blurred / (F_psf + epsilon)
naive_restored = np.real(np.fft.ifft2(F_naive_inv))

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(noisy_blurred, cmap="gray")
plt.axis("off")
plt.title("Blurred + noisy (input)")
plt.subplot(1, 2, 2)
plt.imshow(naive_restored, cmap="gray")
plt.axis("off")
plt.title("Naive inverse filter")
plt.show()
print(f"Naive inverse filter output range: [{naive_restored.min():.1f}, {naive_restored.max():.1f}]")
print("(compare with the original 0-255 range: the noise at frequencies where H is small has blown up)")

The naive inverse filter's output is dominated by noise, wildly outside the original [0, 255] range. Wiener filtering fixes this by adding a small regularization constant `K` before dividing:

`W = H* / (|H|² + K)`

where `H*` is the complex conjugate of `H`. When `K → 0`, this reduces to the naive inverse filter; a larger `K` suppresses noise more (at the cost of leaving more blur in). `K` plays the role of an (inverse) signal-to-noise ratio.

**Ex 10:** Implement the Wiener filter and use it to deconvolve `noisy_blurred`.

In [ ]:
# wiener_deconvolve(F_blurred, F_psf, K) -> restored image (real-valued, spatial domain)
# Tip: the Wiener filter is H* / (|H|^2 + K)
# TODO: your code here


restored_wiener = wiener_deconvolve(F_blurred, F_psf, K=0.05)


def mse(a, b):
    return np.mean((a - b) ** 2)


mse_naive = mse(gray, naive_restored)
mse_wiener = mse(gray, restored_wiener)
print(f"MSE naive inverse filter vs original: {mse_naive:.1f}")
print(f"MSE Wiener filter vs original:        {mse_wiener:.1f}")
assert mse_wiener < mse_naive, "Wiener filtering should beat the naive inverse filter under noise"

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.axis("off")
plt.title("Original")
plt.subplot(1, 3, 2)
plt.imshow(naive_restored, cmap="gray")
plt.axis("off")
plt.title("Naive inverse")
plt.subplot(1, 3, 3)
plt.imshow(restored_wiener, cmap="gray")
plt.axis("off")
plt.title("Wiener (K=0.05)")
plt.show()